In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
# collapse
def rms_envelope(x: np.ndarray, frame_length: int, hop: int) -> np.ndarray:
    """The RMS amplitude of each frame of x, centered on samples 0, hop, 2 * hop, ..."""
    padded = np.pad(x, frame_length // 2)
    frames = np.lib.stride_tricks.sliding_window_view(padded, frame_length)[::hop]
    return np.sqrt(np.mean(frames ** 2, axis=1))

def compress(x: np.ndarray, threshold_db: float, ratio: float,
             key: np.ndarray | None = None, f_s: int = 44100) -> np.ndarray:
    """Turn x down wherever the level of `key` (x itself, by default) exceeds
    the threshold, so that the level above the threshold shrinks by `ratio`."""
    key = x if key is None else key
    hop = f_s // 100                                     # 100 frames per second
    level_db = amplitude_to_db(np.maximum(rms_envelope(key, 4 * hop, hop), 1e-6))
    target_db = np.where(level_db > threshold_db,
                         threshold_db + (level_db - threshold_db) / ratio, level_db)
    gain = db_to_amplitude(target_db - level_db)         # one gain per frame...
    return x * np.interp(np.arange(len(x)), hop * np.arange(len(gain)), gain)  # ...per sample

In [ ]:
f_s = 44100
pad = pq.Audio.from_file("../assets/pad.wav").samples[:, 0]
kick = pq.Audio.from_file("../assets/kick-loop.wav").samples[:, 0]
N = min(len(pad), len(kick))
pad, kick = 0.5 * pad[:N], kick[:N]
ducked = compress(pad, threshold_db=-40.0, ratio=10.0, key=kick)   # the kick turns the pad down
pq.play(pq.Audio(pad + kick, f_s), normalize=True)
pq.play(pq.Audio(ducked + kick, f_s), normalize=True)